# Data Exploration Demo

This notebook builds a small multilingual sample corpus and explores token-length patterns, language-pair differences, and German compound behavior.

In [1]:
from pathlib import Path
import os
import sys
from collections import Counter

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'evaluation').exists():
            return candidate
    raise RuntimeError('Could not locate Machine_Translation project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Nikolai\OneDrive\Desktop\Portfolio\CS_Language_Portfolio\projects\Machine_Translation


In [2]:
sample_rows = [
    {
        'pair': 'en->de',
        'source': 'The company will not increase prices next year.',
        'reference': 'Die Firma wird die Preise naechstes Jahr nicht erhoehen.'
    },
    {
        'pair': 'en->de',
        'source': 'I have never seen this before.',
        'reference': 'Ich habe so etwas noch nie gesehen.'
    },
    {
        'pair': 'de->en',
        'source': 'Der Schmetterlingshaus ist sehr gross.',
        'reference': 'The butterfly house is very large.'
    },
    {
        'pair': 'de->en',
        'source': 'Das Abendessen wird sehr lecker sein.',
        'reference': 'Dinner will be very delicious.'
    },
    {
        'pair': 'ru->en',
        'source': 'Bolshaya koshka spit na myagkom divane.',
        'reference': 'The big cat is sleeping on a soft couch.'
    },
    {
        'pair': 'en->ru',
        'source': 'Machine translation is fascinating.',
        'reference': 'Mashinnyy perevod ochen uvlekatelen.'
    }
]

df = pd.DataFrame(sample_rows)
df['src_tokens'] = df['source'].str.split().str.len()
df['ref_tokens'] = df['reference'].str.split().str.len()
df['length_ratio'] = (df['ref_tokens'] / df['src_tokens']).round(2)

df

,pair,source,reference,src_tokens,ref_tokens,length_ratio
0,en->de,The company will not increase prices next year.,Die Firma wird die Preise naechstes Jahr nicht...,8,9,1.12
1,en->de,I have never seen this before.,Ich habe so etwas noch nie gesehen.,6,7,1.17
2,de->en,Der Schmetterlingshaus ist sehr gross.,The butterfly house is very large.,5,6,1.20
3,de->en,Das Abendessen wird sehr lecker sein.,Dinner will be very delicious.,6,5,0.83
4,ru->en,Bolshaya koshka spit na myagkom divane.,The big cat is sleeping on a soft couch.,6,9,1.50
5,en->ru,Machine translation is fascinating.,Mashinnyy perevod ochen uvlekatelen.,4,4,1.00


In [3]:
pair_stats = (
    df.groupby('pair')
      .agg(
          count=('pair', 'count'),
          avg_src_tokens=('src_tokens', 'mean'),
          avg_ref_tokens=('ref_tokens', 'mean'),
          avg_length_ratio=('length_ratio', 'mean')
      )
      .round(2)
      .sort_values('count', ascending=False)
)

pair_stats

,count,avg_src_tokens,avg_ref_tokens,avg_length_ratio
pair,,,,
de->en,2,5.5,5.5,1.01
en->de,2,7.0,8.0,1.14
en->ru,1,4.0,4.0,1.00
ru->en,1,6.0,9.0,1.50


In [4]:
def likely_compound_tokens(text: str, min_len: int = 10):
    tokens = [t.strip('.,!?;:').lower() for t in text.split()]
    return [t for t in tokens if t.isalpha() and len(t) >= min_len]

de_sources = df[df['pair'] == 'de->en']['source'].tolist()
compound_candidates = []
for sentence in de_sources:
    for token in likely_compound_tokens(sentence):
        compound_candidates.append(token)

Counter(compound_candidates)

Counter({'schmetterlingshaus': 1, 'abendessen': 1})

## Next Steps

- Replace the toy dataset with your real dev/test split.
- Add character-level statistics to inspect script-specific behavior (Latin/Cyrillic).
- Feed explored examples into the evaluation and error taxonomy notebooks.